# Fine-tuning Supervisionado (SFT + LoRA/QLoRA)

Esqueleto do notebook para treinar o adaptador sobre **Qwen/Qwen2.5-Coder-1.5B-Instruct** no Google Colab (ex.: GPU T4), **após** a coleta com o professor (`Qwen/Qwen2.5-14B-Instruct-AWQ`), filtragem (`F1–F5`), medição de baselines e congelamento (`passo1-freeze`).

## Fundamentação Matemática (LoRA e QLoRA)

- **Adaptação de baixo posto (LoRA):**
  $$W' = W + \Delta W, \qquad \Delta W = \frac{\alpha}{r}BA, \qquad A \in \mathbb{R}^{r\times k}, \ B \in \mathbb{R}^{d\times r}, \ r \ll \min(d, k)$$
  onde $W \in \mathbb{R}^{d\times k}$ permanece congelado e $B$ é inicializado em zero (de modo que $\Delta W = 0$ no início do treino).
- **Redução de parâmetros treináveis por matriz:**
  $$\frac{r(d+k)}{dk}$$
  Por exemplo, para $d = k = 2048$ e $r = 16$, temos $\frac{16(2048+2048)}{2048 \times 2048} = \frac{65.536}{4.194.304} \approx 1{,}56\%$.
- **QLoRA (4-bit NF4):** os pesos base ficam quantizados em 4-bit (NF4) e congelados, enquanto os parâmetros do adaptador LoRA são treinados em maior precisão; o ganho principal é na memória de GPU, e não na contagem de parâmetros treináveis do adaptador.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Dependências e Verificação de Ambiente (GPU / Google Drive)

In [2]:
# MINICODE_ROOT_SET
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'max_split_size_mb:128')
from pathlib import Path
for _cand in [Path.cwd(), Path.cwd().parent]:
    if (_cand / 'datasets' / 'sft_jsonl' / 'train.jsonl').exists():
        os.chdir(_cand)
        break
print('Working dir:', Path.cwd())

# Execute no Google Colab se ainda não tiver os pacotes instalados:
# %pip install -q -U torch transformers accelerate peft trl bitsandbytes datasets

import os
import torch

print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM total: {vram_gb:.2f} GB")

for pkg in ["transformers", "peft", "trl", "bitsandbytes"]:
    try:
        mod = __import__(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"{pkg}: NÃO INSTALADO")

# Opcional no Colab: montar Google Drive para salvar checkpoints a cada época
# from google.colab import drive
# drive.mount('/content/drive')


Working dir: /content
CUDA disponível: True
GPU: Tesla T4
VRAM total: 14.56 GB
transformers: 5.22.1
peft: 0.21.2
trl: NÃO INSTALADO
bitsandbytes: NÃO INSTALADO


## 2. Configuração de Hiperparâmetros e Caminhos

In [3]:
# --- Configura??o central consolidada (G7) ---
from dataclasses import dataclass

@dataclass
class TrainConfig:
    model_id: str = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
    train_file: str = "datasets/sft_jsonl/train.jsonl"
    val_file: str = "datasets/sft_jsonl/val.jsonl"
    test_file: str = "datasets/sft_jsonl/test.jsonl"
    output_dir: str = "checkpoints/qwen25_coder_1_5b_lora"
    prompt_version: str = 'minicode-prompt-v2'
    system_prompt: str = 'Você é o Agente Autônomo de Programação do MiniCode.\nSua missão é executar o plano em \'plan.md\' para cumprir os critérios de \'specification.md\'.\n\nA cada turno, responda com UM único objeto JSON, dentro de um bloco ```json:\n{\n  "thought": "raciocínio breve",\n  "tool": "nome_da_ferramenta",\n  "args": { ... }\n}\n\nFerramentas disponíveis (use exatamente estes nomes e argumentos):\n- append_file: Acrescenta conteúdo ao final de um arquivo (cria o arquivo se não existir). Não apaga o conteúdo existente.\n- done: Encerra a implementação e dispara a verificação.\n- list_files: Lista arquivos e diretórios do projeto recursivamente.\n- read_file: Lê o conteúdo de um arquivo de texto com números de linha.\n- replace_text: Substitui um trecho exato de um arquivo. \'old_str\' deve aparecer exatamente uma vez.\n- write_file: Cria ou SUBSTITUI o arquivo inteiro com o conteúdo fornecido. Para acrescentar a um arquivo existente, use append_file.\n- done: encerra a implementação. Sem argumentos. Use quando todos os passos do plano estiverem concluídos. Os testes são executados automaticamente pelo harness depois de \'done\'.\n\nRegras:\n- Use sempre \'path\' para caminhos de arquivo. Não existe \'file_path\'.\n- \'write_file\' SUBSTITUI o arquivo inteiro. Para acrescentar funções ou testes a um arquivo que já existe, use \'append_file\'.\n- Os testes exigidos pela SPEC (test_ac_n) devem existir em tests/ com o nome exato. O harness verifica isso ao final.\n- Não chame \'run_tests\': a verificação é feita pelo harness.\n- Chame \'done\' somente quando todos os passos do plano estiverem concluídos.\n- Não invente argumentos fora da lista acima.\n- Em arquivos de teste, inclua os imports no topo (ex.: `from src.pure_ops import clamp`). Nunca use um módulo ou função sem importá-lo.\n- Arquivos que já existem em tests/ no início são testes da tarefa: NÃO os altere. Coloque os testes \'test_ac_n\' exigidos pela SPEC em um arquivo novo (ex.: tests/test_spec_ac.py), como funções `def test_ac_...` no nível do módulo.\n'
    seed: int = 42
    max_seq_length: int = 2048

    def __post_init__(self):
        from src.agent.orchestrator import montar_system_prompt
        from pathlib import Path
        if not self.system_prompt:
            self.system_prompt = montar_system_prompt(Path.cwd(), catalogo_enxuto=True)

    # QLoRA / LoRA
    use_qlora_4bit: bool = True
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: tuple = (
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    )

    # Treino (SFT)
    num_train_epochs: int = 3
    learning_rate: float = 2e-4
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 16

    # Aliases de compatibilidade para chamadas legadas
    @property
    def model_name(self) -> str:
        return self.model_id

    @property
    def train_path(self) -> str:
        return self.train_file

    @property
    def val_path(self) -> str:
        return self.val_file

    @property
    def test_path(self) -> str:
        return self.test_file

    @property
    def use_4bit(self) -> bool:
        return self.use_qlora_4bit

cfg = TrainConfig()
print(cfg)


TrainConfig(model_id='Qwen/Qwen2.5-Coder-1.5B-Instruct', train_file='datasets/sft_jsonl/train.jsonl', val_file='datasets/sft_jsonl/val.jsonl', test_file='datasets/sft_jsonl/test.jsonl', output_dir='checkpoints/qwen25_coder_1_5b_lora', prompt_version='minicode-prompt-v2', system_prompt='Você é o Agente Autônomo de Programação do MiniCode.\nSua missão é executar o plano em \'plan.md\' para cumprir os critérios de \'specification.md\'.\n\nA cada turno, responda com UM único objeto JSON, dentro de um bloco ```json:\n{\n  "thought": "raciocínio breve",\n  "tool": "nome_da_ferramenta",\n  "args": { ... }\n}\n\nFerramentas disponíveis (use exatamente estes nomes e argumentos):\n- append_file: Acrescenta conteúdo ao final de um arquivo (cria o arquivo se não existir). Não apaga o conteúdo existente.\n- done: Encerra a implementação e dispara a verificação.\n- list_files: Lista arquivos e diretórios do projeto recursivamente.\n- read_file: Lê o conteúdo de um arquivo de texto com números de lin

## 3. Carregamento dos Splits JSONL (`train`, `val`, `test`)

In [4]:
import json
from pathlib import Path

def load_jsonl_records(path_str: str):
    p = Path(path_str)
    if not p.exists():
        print(f"[AVISO] Arquivo não encontrado ainda: {p}")
        return []
    rows = []
    for line in p.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows

train_rows = load_jsonl_records(cfg.train_file)
val_rows = load_jsonl_records(cfg.val_file)
test_rows = load_jsonl_records(cfg.test_file)

print(f"Exemplos carregados -> train: {len(train_rows)}, val: {len(val_rows)}, test: {len(test_rows)}")


Exemplos carregados -> train: 132, val: 43, test: 36


## 4. Tokenização, Template de Chat e Máscara de *Completion Only*

Para o modelo aprender a gerar a ação alvo sem receber *loss* sobre o prompt (`system` + `user`), mascaramos os tokens anteriores à resposta do `assistant` com `-100` nos `labels`.

In [5]:
from transformers import AutoTokenizer

def build_completion_only_example(row: dict, tokenizer, max_length: int = 2048):
    """Formata messages com chat template e mascara prompt (system+user) com -100."""
    messages = row.get("messages", [])
    if len(messages) < 3:
        raise ValueError("Esperado ao menos 3 mensagens: system, user e assistant.")

    prompt_messages = messages[:-1]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_ids = tokenizer(prompt_text, truncation=True, max_length=max_length, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, truncation=True, max_length=max_length, add_special_tokens=False)["input_ids"]

    prompt_len = min(len(prompt_ids), len(full_ids))
    labels = [-100] * prompt_len + full_ids[prompt_len:]
    attention_mask = [1] * len(full_ids)

    return {
        "input_ids": full_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

# Exemplo de uso após carregar o tokenizer:
# tokenizer = AutoTokenizer.from_pretrained(cfg.model_id)
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token


## 5. Carregamento do Modelo Base e Configuração LoRA / QLoRA

In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

def load_lora_model_and_tokenizer(cfg: TrainConfig):
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {"device_map": "auto"}
    if cfg.use_qlora_4bit:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    else:
        model_kwargs["torch_dtype"] = torch.float16

    model = AutoModelForCausalLM.from_pretrained(cfg.model_id, **model_kwargs)
    model.config.use_cache = False

    if cfg.use_qlora_4bit:
        model = prepare_model_for_kbit_training(model)

    lora_cfg = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.target_modules),
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model, tokenizer


In [7]:
# TrainConfig consolidada na c?lula anterior (G7)
assert isinstance(cfg, TrainConfig)


In [8]:
# --- Carregamento JSONL e máscara de Completion Only ---
import json
import torch
from datasets import Dataset

def load_jsonl(path: str) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def make_completion_only_formatter(tokenizer, max_length: int):
    """
    Espera cada registro com chave 'messages':
    [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", "content": "..."}]
    Calcula loss apenas nos tokens da resposta do assistant (prompt -> -100).
    """
    def format_example(example: dict) -> dict:
        messages = example["messages"]
        prompt_messages = messages[:-1]

        full_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        prompt_text = tokenizer.apply_chat_template(
            prompt_messages, tokenize=False, add_generation_prompt=True
        )

        full_ids = tokenizer(
            full_text, truncation=True, max_length=max_length, add_special_tokens=False
        )["input_ids"]
        prompt_ids = tokenizer(
            prompt_text, truncation=True, max_length=max_length, add_special_tokens=False
        )["input_ids"]

        prompt_len = min(len(prompt_ids), len(full_ids))
        labels = [-100] * prompt_len + full_ids[prompt_len:]

        return {
            "input_ids": full_ids,
            "attention_mask": [1] * len(full_ids),
            "labels": labels,
        }

    return format_example


In [10]:
# Instala as dependências necessárias no Colab se não estiverem presentes
%pip install -q trl bitsandbytes

# ---  Modelo Base (4-bit QLoRA) + LoRA + SFTTrainer ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

def build_model_and_tokenizer(cfg: TrainConfig):
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = None
    torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    if cfg.use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch_dtype,
        )

    model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        quantization_config=bnb_config,
        dtype=torch_dtype,
        device_map="auto",
    )

    model.config.use_cache = False
    if cfg.use_4bit:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    lora_config = LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.target_modules),
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model, tokenizer

# Exemplo de SFTConfig salvando checkpoints a cada época:
# sft_args = SFTConfig(
#     output_dir=cfg.output_dir,
#     max_length=cfg.max_seq_length,
#     optim='paged_adamw_8bit',
#     num_train_epochs=cfg.num_train_epochs,
#     learning_rate=cfg.learning_rate,
#     per_device_train_batch_size=cfg.per_device_train_batch_size,
#     per_device_eval_batch_size=cfg.per_device_eval_batch_size,
#     gradient_accumulation_steps=cfg.gradient_accumulation_steps,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     save_total_limit=cfg.num_train_epochs,
#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     greater_is_better=False,
#     logging_steps=10,
#     fp16=(torch_dtype == torch.float16),
#     bf16=(torch_dtype == torch.bfloat16),
#     seed=cfg.seed,
#     dataset_num_proc=1,
# )

In [16]:
import gc, json as _json, torch
from pathlib import Path
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

for _v in ['model', 'trainer', 'base_model']:
    if _v in globals():
        del globals()[_v]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# --- Execução do Fine-Tuning QLoRA (G7) em FP16/FP32 nativo da T4 + Salvamento do Adapter ---
assert len(train_rows) > 0, "train.jsonl vazio!"
assert len(val_rows) > 0, "val.jsonl vazio!"
for r in train_rows[:3]:
    assert r["messages"][0]["content"] == cfg.system_prompt, "Divergência de system_prompt (G3)!"

model_id = (
    getattr(cfg, "model_name_or_path", None)
    or getattr(cfg, "model_id", None)
    or getattr(cfg, "base_model", None)
    or getattr(cfg, "model_name", None)
    or "Qwen/Qwen2.5-Coder-1.5B-Instruct"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=getattr(cfg, "lora_r", 16),
    lora_alpha=getattr(cfg, "lora_alpha", 32),
    lora_dropout=getattr(cfg, "lora_dropout", 0.05),
    target_modules=list(getattr(
        cfg,
        "target_modules",
        ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )),
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

# Elimina qualquer BFloat16 residual (T4 compute capability 7.5 não suporta BF16)
for mod in model.modules():
    if getattr(mod, "compute_dtype", None) == torch.bfloat16:
        mod.compute_dtype = torch.float16
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)
    elif p.dtype == torch.bfloat16:
        p.data = p.data.to(torch.float16)
for b in model.buffers():
    if b.dtype == torch.bfloat16:
        b.data = b.data.to(torch.float16)
model.config.dtype = torch.float16
model.config.torch_dtype = torch.float16

print("Dtypes ativos no modelo:", {p.dtype for p in model.parameters()})
model.print_trainable_parameters()

formatter = make_completion_only_formatter(tokenizer, cfg.max_seq_length)
train_ds = Dataset.from_list(train_rows).map(formatter, remove_columns=list(train_rows[0].keys()))
val_ds = Dataset.from_list(val_rows).map(formatter, remove_columns=list(val_rows[0].keys()))

sft_args = SFTConfig(
    output_dir=cfg.output_dir,
    max_length=cfg.max_seq_length,
    optim="paged_adamw_8bit",
    num_train_epochs=cfg.num_train_epochs,
    learning_rate=cfg.learning_rate,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=5,
    fp16=False,
    bf16=False,
    seed=cfg.seed,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_num_proc=1,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, pad_to_multiple_of=8),
)

train_result = trainer.train()
final_dir = Path(cfg.output_dir) / "final_adapter"
final_dir.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print("Treino concluído! Adapter salvo em:", final_dir.resolve())
print("Métricas de treino:", train_result.metrics)

metrics_out = Path(cfg.output_dir) / "training_metrics.json"
metrics_out.write_text(
    _json.dumps({"train_metrics": train_result.metrics, "log_history": trainer.state.log_history}, indent=2),
    encoding="utf-8",
)
print("Histórico salvo em:", metrics_out.resolve())

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Dtypes ativos no modelo: {torch.uint8, torch.float32}
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Map:   0%|          | 0/132 [00:00<?, ? examples/s]

Map:   0%|          | 0/43 [00:00<?, ? examples/s]

Truncating train dataset (num_proc=1):   0%|          | 0/132 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset (num_proc=1):   0%|          | 0/132 [00:00<?, ? examples/s]

Truncating eval dataset (num_proc=1):   0%|          | 0/43 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset (num_proc=1):   0%|          | 0/43 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.420468,0.169980,0.178841,0.939996,156547.000000
2,0.126863,0.123094,0.135849,0.945288,313094.000000
3,0.086943,0.129768,0.115710,0.946552,469641.000000


Treino concluído! Adapter salvo em: /content/checkpoints/qwen25_coder_1_5b_lora/final_adapter
Métricas de treino: {'train_runtime': 968.7706, 'train_samples_per_second': 0.409, 'train_steps_per_second': 0.028, 'total_flos': 3756734776369152.0, 'train_loss': 0.17307655899612992, 'epoch': 3.0}
Histórico salvo em: /content/checkpoints/qwen25_coder_1_5b_lora/training_metrics.json


In [17]:
# Compacta adapter LoRA e training_metrics.json para download direto do Colab
import shutil
from pathlib import Path
ckpt_dir = Path(cfg.output_dir)
if ckpt_dir.exists():
    archive = shutil.make_archive('qwen25_coder_1_5b_lora', 'zip', root_dir='checkpoints', base_dir='qwen25_coder_1_5b_lora')
    print('Arquivo pronto para download:', archive)


Arquivo pronto para download: /content/qwen25_coder_1_5b_lora.zip
